# Inquiro World Model Demo

This notebook demonstrates the core functionality of the Inquiro World Model - 
the central knowledge store for our autonomous AI scientist.

## What You'll Learn
1. Creating and storing findings
2. Building relationships between findings
3. Managing hypotheses
4. Querying the knowledge graph
5. Generating summaries for LLMs

## Setup

First, let's import our modules and create a fresh World Model.

In [1]:
# Add project root to path
import sys
sys.path.insert(0, '..')

from src.world_model.world_model import WorldModel
from src.world_model.models import FindingType, RelationshipType, HypothesisStatus

# Create a fresh world model for this demo
wm = WorldModel('../data/demo_world_model.db')
print('✅ World Model initialized!')

✅ World Model initialized!


## 1. Adding Findings

Findings are the atomic units of knowledge. Each finding represents a scientific claim
with its source and confidence level.

### Finding Types:
- `data_analysis` - Results from code execution
- `literature` - Information from papers
- `interpretation` - Conclusions drawn from other findings

In [2]:
# Add a data analysis finding
f1 = wm.add_finding(
    claim="47 metabolites are significantly altered in hypothermia (FDR < 0.05)",
    finding_type="data_analysis",
    source={
        "type": "notebook",
        "path": "analysis/metabolomics_r1.ipynb",
        "cell": 15
    },
    cycle=1,
    confidence=0.92,
    tags=["metabolomics", "differential_expression", "hypothermia"],
    statistical_support={
        "n_significant": 47,
        "threshold": "FDR<0.05",
        "method": "t-test with BH correction"
    }
)
print(f'Created finding: {f1}')

Created finding: f_ce313227


In [3]:
# Add a literature finding
f2 = wm.add_finding(
    claim="Nucleotide salvage pathways are activated during cellular stress as a protective mechanism",
    finding_type="literature",
    source={
        "type": "paper",
        "doi": "10.1038/s41586-021-03819-2",
        "title": "Metabolic adaptation in stress response",
        "authors": ["Smith J", "Johnson A", "Williams B"],
        "year": 2021
    },
    cycle=1,
    confidence=0.95,
    tags=["nucleotide_salvage", "stress_response", "neuroprotection"]
)
print(f'Created finding: {f2}')

Created finding: f_bb38b1ea


In [4]:
# Add more findings to build a rich knowledge base
f3 = wm.add_finding(
    claim="Hypoxanthine levels increased 2.3-fold in hypothermia group",
    finding_type="data_analysis",
    source={"type": "notebook", "path": "analysis/metabolomics_r2.ipynb", "cell": 8},
    cycle=2,
    confidence=0.88,
    tags=["metabolomics", "hypoxanthine", "purine"]
)

f4 = wm.add_finding(
    claim="Hypoxanthine is a key intermediate in nucleotide salvage",
    finding_type="literature",
    source={"type": "paper", "doi": "10.1016/j.cell.2020.01.001", "title": "Purine metabolism review", "year": 2020},
    cycle=2,
    confidence=0.98,
    tags=["hypoxanthine", "nucleotide_salvage", "purine"]
)

f5 = wm.add_finding(
    claim="Therapeutic hypothermia may activate nucleotide salvage for neuroprotection",
    finding_type="interpretation",
    source={"type": "notebook", "path": "analysis/interpretation_r2.ipynb", "cell": 5},
    cycle=2,
    confidence=0.75,
    tags=["hypothesis", "mechanism", "neuroprotection"]
)

print(f'Created findings: {f3}, {f4}, {f5}')

Created findings: f_55cc3124, f_92961bf8, f_f87d96c1


## 2. Creating Relationships

Relationships connect findings to build a knowledge graph.

### Relationship Types:
- `supports` - Evidence supports a conclusion
- `contradicts` - Findings conflict with each other
- `extends` - One finding builds on another
- `relates_to` - General association

In [5]:
# Build the evidence network
wm.add_relationship(f1, f5, "supports", strength=0.8, 
                    reasoning="Metabolite changes include nucleotide pathway components")

wm.add_relationship(f2, f5, "supports", strength=0.9,
                    reasoning="Literature confirms salvage activation in stress")

wm.add_relationship(f3, f5, "supports", strength=0.85,
                    reasoning="Hypoxanthine increase indicates salvage activation")

wm.add_relationship(f4, f3, "extends", strength=0.95,
                    reasoning="Explains the biochemical significance of hypoxanthine")

print('✅ Relationships created!')

✅ Relationships created!


## 3. Adding Hypotheses

Hypotheses are higher-level claims that we track throughout the research process.

In [6]:
h1 = wm.add_hypothesis(
    statement="Therapeutic hypothermia activates nucleotide salvage pathways as a neuroprotective mechanism",
    cycle=2,
    supporting_finding_ids=[f1, f2, f3, f5]
)
print(f'Created hypothesis: {h1}')

# Check hypothesis status
hyp = wm.get_hypothesis(h1)
print(f'Status: {hyp.status}')
print(f'Supporting findings: {len(hyp.supporting_finding_ids)}')

Created hypothesis: h_e5db143f
Status: proposed
Supporting findings: 4


## 4. Querying the Knowledge Graph

In [7]:
# Get findings that support our interpretation
supporters = wm.get_supporting_findings(f5)
print(f'\nFindings supporting the interpretation (f5):')
for s in supporters:
    print(f'  [{s.id}] {s.claim[:60]}...')


Findings supporting the interpretation (f5):
  [f_ce313227] 47 metabolites are significantly altered in hypothermia (FDR...
  [f_bb38b1ea] Nucleotide salvage pathways are activated during cellular st...
  [f_55cc3124] Hypoxanthine levels increased 2.3-fold in hypothermia group...


In [8]:
# Get evidence chain
print(f'\nEvidence chain for interpretation:')
chain = wm.get_evidence_chain(f5)
for item in chain:
    indent = '  ' * item['depth']
    print(f"{indent}[Depth {item['depth']}] {item['claim'][:50]}...")


Evidence chain for interpretation:
[Depth 0] Therapeutic hypothermia may activate nucleotide sa...
  [Depth 1] 47 metabolites are significantly altered in hypoth...
  [Depth 1] Nucleotide salvage pathways are activated during c...
  [Depth 1] Hypoxanthine levels increased 2.3-fold in hypother...


In [9]:
# Find unexplored areas
unexplored = wm.get_unexplored_findings()
print(f'\nUnexplored findings (potential research directions):')
for f in unexplored:
    print(f'  [{f.id}] {f.claim[:60]}...')


Unexplored findings (potential research directions):
  [f_f87d96c1] Therapeutic hypothermia may activate nucleotide salvage for ...


In [10]:
# Get top supported findings
top = wm.get_top_findings(3)
print(f'\nTop supported findings:')
for item in top:
    f = item['finding']
    print(f"  [{f.id}] Support: {item['support_count']}, Conf: {f.confidence:.2f}")
    print(f"      {f.claim[:60]}...")


Top supported findings:
  [f_f87d96c1] Support: 3, Conf: 0.75
      Therapeutic hypothermia may activate nucleotide salvage for ...
  [f_92961bf8] Support: 0, Conf: 0.98
      Hypoxanthine is a key intermediate in nucleotide salvage...
  [f_bb38b1ea] Support: 0, Conf: 0.95
      Nucleotide salvage pathways are activated during cellular st...


## 5. Generating LLM Summary

This summary is what gets passed to the orchestrator and agents
so they understand the current state of knowledge.

In [11]:
summary = wm.get_summary()
print(summary)

CURRENT KNOWLEDGE STATE

Cycle: 2
Total Findings: 5
Total Relationships: 4

RECENT FINDINGS:
----------------------------------------
[f_f87d96c1] (cycle 2, notebook, conf=0.75)
    Therapeutic hypothermia may activate nucleotide salvage for neuroprotection

[f_92961bf8] (cycle 2, paper, conf=0.98)
    Hypoxanthine is a key intermediate in nucleotide salvage

[f_55cc3124] (cycle 2, notebook, conf=0.88)
    Hypoxanthine levels increased 2.3-fold in hypothermia group

[f_bb38b1ea] (cycle 1, paper, conf=0.95)
    Nucleotide salvage pathways are activated during cellular stress as a protective mechanism

[f_ce313227] (cycle 1, notebook, conf=0.92)
    47 metabolites are significantly altered in hypothermia (FDR < 0.05)

ACTIVE HYPOTHESES:
----------------------------------------
[h_e5db143f] (proposed)
    Therapeutic hypothermia activates nucleotide salvage pathways as a neuroprotective mechanism
    Supported by: f_ce313227, f_bb38b1ea, f_55cc3124, f_f87d96c1

UNEXPLORED AREAS (findings 

## 6. Statistics

In [12]:
stats = wm.get_statistics()
print('World Model Statistics:')
for key, value in stats.items():
    print(f'  {key}: {value}')

World Model Statistics:
  total_findings: 5
  findings_by_type: {'data_analysis': 2, 'interpretation': 1, 'literature': 2}
  total_relationships: 4
  hypotheses_by_status: {'proposed': 1}
  tasks_by_status: {}
  current_cycle: 2


## 7. Visualize the Knowledge Graph (Optional)

If you have matplotlib installed, you can visualize the graph.

In [13]:
try:
    import matplotlib.pyplot as plt
    import networkx as nx
    
    # Create figure
    plt.figure(figsize=(12, 8))
    
    # Get graph
    G = wm.graph
    
    # Color nodes by type
    colors = []
    for node in G.nodes():
        finding = wm.get_finding(node)
        if finding:
            if finding.finding_type == 'data_analysis':
                colors.append('lightblue')
            elif finding.finding_type == 'literature':
                colors.append('lightgreen')
            else:
                colors.append('lightyellow')
        else:
            colors.append('gray')
    
    # Draw
    pos = nx.spring_layout(G, seed=42)
    nx.draw(G, pos, node_color=colors, with_labels=True, 
            node_size=2000, font_size=8, arrows=True)
    
    # Add legend
    plt.legend(['Data Analysis', 'Literature', 'Interpretation'], 
               loc='upper left')
    plt.title('Inquiro Knowledge Graph')
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print('Install matplotlib for visualization: pip install matplotlib')

Install matplotlib for visualization: pip install matplotlib


## Cleanup

In [14]:
wm.close()
print('✅ Demo complete! World Model closed.')

✅ Demo complete! World Model closed.


---

## Summary

In this demo, we:

1. **Created findings** from data analysis and literature
2. **Built relationships** to form a knowledge graph
3. **Tracked hypotheses** through the research process
4. **Queried** for supporting evidence and unexplored areas
5. **Generated summaries** for LLM context

This World Model is the foundation that all Inquiro agents will read from and write to!